## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q umap-learn

In [ ]:
import os, glob, json, csv, random, time, urllib.request
from collections import defaultdict
from scipy.io import loadmat
from google.colab import drive
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import umap

In [ ]:
BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/MyDrive/maxwell-braid"
INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
OUTPUT_ROOT = f"{DRIVE_PROJECT_DIR}/outputs"
SUBJECT = "subj01"
BETA_DIR = f"{BASE}/subject01_visual_brain_responses"
CLIP_DIR = f"{BASE}/subject01_clip_image_embeddings"
EXP = f"{BASE}/nsd_expdesign.mat"
TRAINING_LOSS_FUNCTIONS = ["mse", "cos", "mse_cos", "contrastive", "bidir_clip", "silhouette"]
EVAL_LOSS_FUNCTIONS = ["mse", "cos", "silhouette"]
SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 15
BATCH = 512
LR = 1e-4
DROPOUT = 0.5
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"{OUTPUT_ROOT}/{TIMESTAMP}"
SILHOUETTE_MAX = 1000
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
run_config = {"timestamp": TIMESTAMP, "output_dir": OUTPUT_DIR, "subject": SUBJECT, "training_loss_functions": TRAINING_LOSS_FUNCTIONS, "eval_loss_functions": EVAL_LOSS_FUNCTIONS, "seed": SEED, "epochs": EPOCHS, "batch": BATCH, "lr": LR, "dropout": DROPOUT, "silhouette_max": SILHOUETTE_MAX, "input_dir": INPUT_DIR}
with open(f"{OUTPUT_DIR}/run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)


In [ ]:
!cp -r {INPUT_DIR}/subject01_visual_brain_responses /content/
!cp -r {INPUT_DIR}/subject01_clip_image_embeddings /content/

In [ ]:
if not os.path.exists(EXP):
    urllib.request.urlretrieve("https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)

def session_id(path):
    return int(''.join(c for c in os.path.basename(path)[-6:] if c.isdigit()))

beta_sess = {session_id(p): p for p in glob.glob(f"{BETA_DIR}/{SUBJECT}_visualroi_session*.pt")}
clip_sess = {session_id(p): p for p in glob.glob(f"{CLIP_DIR}/{SUBJECT}_clip_embeds*.pt")}
sessions = sorted(set(beta_sess) & set(clip_sess))
betas = torch.cat([torch.load(beta_sess[s]).float() for s in sessions])
clips = torch.cat([torch.load(clip_sess[s]).float() for s in sessions])
mat = loadmat(EXP)
masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
imgbrick_ids = subjectim[int(SUBJECT[-2:]) - 1, masterordering]
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)
img_of = np.array([int(imgbrick_ids[(sessions[g // 750] - 1) * 750 + g % 750]) for g in range(len(betas))])
is_test = np.array([i in shared_ids for i in img_of])
rest_img = np.array(sorted(set(img_of[~is_test])))
np.random.RandomState(SEED).shuffle(rest_img)
val_img = set(rest_img[:int(0.05 * len(rest_img))])
in_val = np.array([i in val_img for i in img_of])
train_idx = np.where(~is_test & ~in_val)[0]
val_idx = np.where(~is_test & in_val)[0]
test_idx = np.where(is_test)[0]
beta_mean, beta_std = betas[train_idx].mean(0), betas[train_idx].std(0) + 1e-6
clip_mean, clip_std = clips[train_idx].mean(0), clips[train_idx].std(0) + 1e-6

def norm(b, c):
    return (b - beta_mean) / beta_std, (c - clip_mean) / clip_std

groups = defaultdict(list)
for g in test_idx:
    groups[int(img_of[g])].append(g)
test_img_ids = list(groups)
test_betas = torch.stack([betas[gs].mean(0) for gs in groups.values()])
test_clips = torch.stack([clips[gs[0]] for gs in groups.values()])
train_ds = TensorDataset(*norm(betas[train_idx], clips[train_idx]))
val_ds = TensorDataset(*norm(betas[val_idx], clips[val_idx]))
test_ds = TensorDataset(*norm(test_betas, test_clips))
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, generator=loader_generator)
train_eval_loader = DataLoader(train_ds, batch_size=BATCH)
val_loader = DataLoader(val_ds, batch_size=BATCH)
test_loader = DataLoader(test_ds, batch_size=BATCH)
print(f"device={DEVICE} output={OUTPUT_DIR}")
print(f"sessions={sessions} betas={tuple(betas.shape)} clips={tuple(clips.shape)}")
print(f"train={len(train_idx)} val={len(val_idx)} test_trials={len(test_idx)} test_images={len(test_img_ids)}")


## Model & Loss Functions


In [ ]:
class FMRIEncoderMLP(nn.Module):
    def __init__(self, input_dim, output_dim=1280, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 2048), nn.LayerNorm(2048), nn.ReLU(), nn.Dropout(dropout), nn.Linear(2048, 4096), nn.LayerNorm(4096), nn.ReLU(), nn.Dropout(dropout), nn.Linear(4096, output_dim))
    def forward(self, x):
        return self.net(x)

def loss_fn(name):
    if name == "mse":
        return nn.MSELoss()
    if name == "cos":
        return lambda pred, y: 1 - F.cosine_similarity(pred, y, dim=1).mean()
    if name == "mse_cos":
        return lambda pred, y: F.mse_loss(pred, y) + 1 - F.cosine_similarity(pred, y, dim=1).mean()
    if name == "contrastive":
        def fn(pred, y):
            pred = F.normalize(pred, dim=1)
            y = F.normalize(y, dim=1)
            logits = pred @ y.T / 0.07
            labels = torch.arange(len(pred), device=pred.device)
            return F.cross_entropy(logits, labels)
        return fn
    if name == "bidir_clip":
        def fn(pred, y):
            pred = F.normalize(pred, dim=1)
            y = F.normalize(y, dim=1)
            logits = pred @ y.T / 0.07
            labels = torch.arange(len(pred), device=pred.device)
            return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))
        return fn
    if name == "silhouette":
        def fn(pred, y):
            z = torch.cat([pred, y], dim=0)
            labels = torch.cat([torch.zeros(len(pred), device=z.device), torch.ones(len(y), device=z.device)])
            d = torch.cdist(z, z)
            same = labels[:, None] == labels[None, :]
            other = ~same
            eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
            same = same & ~eye
            a = (d * same).sum(1) / same.sum(1).clamp_min(1)
            b = (d * other).sum(1) / other.sum(1).clamp_min(1)
            return ((b - a) / torch.maximum(a, b).clamp_min(1e-6)).mean()
        return fn
    raise ValueError(name)


## Training

In [ ]:
def collect(model, loader, raw=False):
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for x, y in loader:
            pred = model(x.to(DEVICE))
            true = y.to(DEVICE)
            if raw:
                pred = pred * clip_std.to(DEVICE) + clip_mean.to(DEVICE)
                true = true * clip_std.to(DEVICE) + clip_mean.to(DEVICE)
            P.append(pred.cpu())
            T.append(true.cpu())
    return torch.cat(P), torch.cat(T)

def metric_value(name, pred, true):
    if name == "mse":
        return float(F.mse_loss(pred, true))
    if name == "cos":
        return float(1 - F.cosine_similarity(pred, true, dim=1).mean())
    if name == "silhouette":
        x = torch.cat([pred, true]).numpy()
        y = np.r_[np.zeros(len(pred)), np.ones(len(true))]
        if len(x) > SILHOUETTE_MAX:
            ids = np.random.RandomState(0).choice(len(x), SILHOUETTE_MAX, replace=False)
            x, y = x[ids], y[ids]
        return float(silhouette_score(x, y))
    raise ValueError(name)

def evaluate_metrics(model, loader):
    pred, true = collect(model, loader)
    return {name: metric_value(name, pred, true) for name in EVAL_LOSS_FUNCTIONS}

results = {}
for train_loss in TRAINING_LOSS_FUNCTIONS:
    model_dir = f"{OUTPUT_DIR}/model_{train_loss}"
    os.makedirs(model_dir, exist_ok=True)
    log_path = f"{model_dir}/training_log.jsonl"
    open(log_path, "w").close()
    model = FMRIEncoderMLP(betas.shape[1], clips.shape[1], DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    objective = loss_fn(train_loss)
    history = {split: {metric: [] for metric in EVAL_LOSS_FUNCTIONS} for split in ["train", "val"]}
    print(f"\n=== train_loss={train_loss} model_dir={model_dir} ===")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = objective(model(x), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_scores = evaluate_metrics(model, train_eval_loader)
        val_scores = evaluate_metrics(model, val_loader)
        for metric in EVAL_LOSS_FUNCTIONS:
            history["train"][metric].append(train_scores[metric])
            history["val"][metric].append(val_scores[metric])
        row = {"epoch": epoch, "train_loss_function": train_loss}
        for m in EVAL_LOSS_FUNCTIONS:
            row[f"train_{m}"] = train_scores[m]
            row[f"val_{m}"] = val_scores[m]
        with open(log_path, "a") as f:
            f.write(json.dumps(row) + "\n")
        with open(f"{model_dir}/history.json", "w") as f:
            json.dump(history, f, indent=2)
        values = []
        for m in EVAL_LOSS_FUNCTIONS:
            values.append(f"train_{m}={train_scores[m]:.4f}")
            values.append(f"val_{m}={val_scores[m]:.4f}")
        print(f"epoch={epoch:05d} " + " ".join(values))
    torch.save(model.state_dict(), f"{model_dir}/model.pth")
    train_summary = {"train_loss_function": train_loss, "model_dir": model_dir, "model_path": f"{model_dir}/model.pth", "history_path": f"{model_dir}/history.json", "training_log_path": log_path, "final_train": {m: history["train"][m][-1] for m in EVAL_LOSS_FUNCTIONS}, "final_val": {m: history["val"][m][-1] for m in EVAL_LOSS_FUNCTIONS}}
    with open(f"{model_dir}/train_summary.json", "w") as f:
        json.dump(train_summary, f, indent=2)
    for metric in EVAL_LOSS_FUNCTIONS:
        plt.figure(dpi=180)
        plt.plot(history["train"][metric], label="train")
        plt.plot(history["val"][metric], label="val")
        plt.xlabel("epoch")
        plt.ylabel(metric)
        plt.title(f"train loss={train_loss} eval={metric}")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.savefig(f"{model_dir}/train_val_{metric}.png", dpi=180, bbox_inches="tight")
        plt.show()
    results[train_loss] = {"model": model, "model_dir": model_dir, "history": history, "train_summary": train_summary}


## Evaluation

In [ ]:
summary_rows = []
summary_fields = ["train_loss_function", "dataset_split", *EVAL_LOSS_FUNCTIONS]
for train_loss, result in results.items():
    model_dir = result["model_dir"]
    model = FMRIEncoderMLP(betas.shape[1], clips.shape[1], DROPOUT).to(DEVICE)
    model.load_state_dict(torch.load(f"{model_dir}/model.pth", map_location=DEVICE))
    test_pred, test_true = collect(model, test_loader)
    test_scores = {name: metric_value(name, test_pred, test_true) for name in EVAL_LOSS_FUNCTIONS}
    split_rows = [
        {"train_loss_function": train_loss, "dataset_split": "train", **{m: result["history"]["train"][m][-1] for m in EVAL_LOSS_FUNCTIONS}},
        {"train_loss_function": train_loss, "dataset_split": "val", **{m: result["history"]["val"][m][-1] for m in EVAL_LOSS_FUNCTIONS}},
        {"train_loss_function": train_loss, "dataset_split": "test", **test_scores},
    ]
    summary_rows.extend(split_rows)
    with open(f"{model_dir}/eval_results.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=summary_fields)
        writer.writeheader()
        writer.writerows(split_rows)
    print(f"\n=== eval_model={train_loss} model_dir={model_dir} ===")
    for row in split_rows:
        print(" ".join([f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items()]))
    pred, true = collect(model, test_loader, raw=True)
    pred_np, true_np = pred.numpy(), true.numpy()
    emb = np.concatenate([pred_np, true_np])
    colors = ["green"] * len(pred_np) + ["blue"] * len(true_np)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150)
    pca = PCA(n_components=2).fit_transform(emb)
    tsne = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(emb)
    umap_xy = umap.UMAP(n_components=2, random_state=0).fit_transform(emb)
    for ax, xy, title in zip(axes, [pca, tsne, umap_xy], ["PCA", "t-SNE", "UMAP"]):
        ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=6, alpha=0.5)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[-1].legend([plt.Line2D([], [], marker="o", ls="", color="green"), plt.Line2D([], [], marker="o", ls="", color="blue")], ["fMRI-predicted", "CLIP image"])
    fig.suptitle(f"test manifold train loss={train_loss}")
    plt.tight_layout()
    plt.savefig(f"{model_dir}/eval_manifold.png", dpi=150, bbox_inches="tight")
    plt.show()
    K = 50
    V = PCA(n_components=K).fit(clips[train_idx].numpy()).components_
    def cum_var(x):
        xc = x - x.mean(0)
        return np.cumsum(((xc @ V.T) ** 2).sum(0) / len(xc)) / ((xc ** 2).sum() / len(xc))
    ks = np.arange(1, K + 1)
    plt.figure(figsize=(9, 5), dpi=180)
    plt.plot(ks, cum_var(clips[train_idx].numpy()), "o-", ms=3, label="train true CLIP")
    plt.plot(ks, cum_var(test_clips.numpy()), "s-", ms=3, label="test true CLIP")
    plt.plot(ks, cum_var(pred_np), "s-", ms=3, label="fMRI-predicted")
    plt.xlabel("principal components")
    plt.ylabel("cumulative variance")
    plt.ylim(0, 1)
    plt.grid(alpha=0.3, ls="--")
    plt.legend()
    plt.title(f"variance train loss={train_loss}")
    plt.savefig(f"{model_dir}/eval_scree.png", dpi=180, bbox_inches="tight")
    plt.show()
with open(f"{OUTPUT_DIR}/eval_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=summary_fields)
    writer.writeheader()
    writer.writerows(summary_rows)
print(f"\neval summary saved to {OUTPUT_DIR}/eval_summary.csv")


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

consolidated_rows = []

def l2_normalize(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def mean_center(x):
    return x - x.mean(axis=0, keepdims=True)

def pairwise_sq_dists(a, b):
    a_sq = (a ** 2).sum(axis=1, keepdims=True)
    b_sq = (b ** 2).sum(axis=1, keepdims=True).T
    return np.clip(a_sq + b_sq - 2 * a @ b.T, a_min=0, a_max=None)

def mmd_gaussian(x, y):
    z = np.concatenate([x, y])
    rng = np.random.RandomState(SEED)
    sub = z[rng.choice(len(z), size=min(len(z), 500), replace=False)]
    d2 = pairwise_sq_dists(sub, sub)
    gamma = 1.0 / (2 * np.median(d2[d2 > 0]))
    kxx = np.exp(-gamma * pairwise_sq_dists(x, x))
    kyy = np.exp(-gamma * pairwise_sq_dists(y, y))
    kxy = np.exp(-gamma * pairwise_sq_dists(x, y))
    m, n = len(x), len(y)
    return float((kxx.sum() - np.trace(kxx)) / (m * (m - 1)) + (kyy.sum() - np.trace(kyy)) / (n * (n - 1)) - 2 * kxy.mean())

def top1_retrieval(pred, true, n_loops=30, n_samples=300):
    rng = np.random.RandomState(SEED)
    pred = F.normalize(pred.to(DEVICE), dim=1)
    true = F.normalize(true.to(DEVICE), dim=1)
    fwd, bwd = [], []
    for _ in range(n_loops):
        idx = rng.choice(len(true), size=min(n_samples, len(true)), replace=False)
        idx = torch.tensor(idx, device=DEVICE)
        labels = torch.arange(len(idx), device=DEVICE)
        sim = pred[idx] @ true[idx].T
        fwd.append((sim.argmax(1) == labels).float().mean().item())
        bwd.append((sim.argmax(0) == labels).float().mean().item())
    return float(np.mean(fwd)), float(np.mean(bwd))

for train_loss, result in results.items():
    model = result["model"]
    model.eval()
    pred, true = collect(model, test_loader, raw=True)
    pred_np, true_np = pred.numpy(), true.numpy()
    x = np.concatenate([pred_np, true_np])
    y = np.array([0] * len(pred_np) + [1] * len(true_np))
    row = {
        "train_loss_function": train_loss,
        "mse_loss": float(F.mse_loss(pred, true)),
        "cosine_loss": float(1 - F.cosine_similarity(pred, true, dim=1).mean()),
        "cosine_similarity": float(F.cosine_similarity(pred, true, dim=1).mean()),
        "silhouette": float(silhouette_score(x, y, metric="cosine")),
        "domain_logreg_acc": float(cross_val_score(LogisticRegression(max_iter=1000), x, y, cv=5).mean()),
    }
    for label, pred_view, true_view in [("raw", pred_np, true_np), ("l2_normalized", l2_normalize(pred_np), l2_normalize(true_np)), ("mean_centered", mean_center(pred_np), mean_center(true_np))]:
        view_x = np.concatenate([pred_view, true_view])
        view_y = np.array([0] * len(pred_view) + [1] * len(true_view))
        row[f"c2st_logreg_{label}"] = float(cross_val_score(LogisticRegression(max_iter=1000), view_x, view_y, cv=5).mean())
        row[f"c2st_rbfsvm_{label}"] = float(cross_val_score(SVC(kernel="rbf"), view_x, view_y, cv=5).mean())
    row["mmd"] = mmd_gaussian(pred_np, true_np)
    row["forward_retrieval_top1"], row["backward_retrieval_top1"] = top1_retrieval(pred, true)
    consolidated_rows.append(row)

eval_table = pd.DataFrame(consolidated_rows).sort_values("mse_loss")
eval_table.to_csv(f"{OUTPUT_DIR}/consolidated_eval_metrics.csv", index=False)
display(eval_table)
print(f"saved={OUTPUT_DIR}/consolidated_eval_metrics.csv")
